In [0]:
dbutils.widgets.text("event_blob_path", "")

event_blob_path = dbutils.widgets.get("event_blob_path")

if not event_blob_path:
    raise ValueError(
        "event_blob_path is empty. Run through ADF."
    )

print("Event blob path:", event_blob_path)

In [0]:
relative_path = event_blob_path.replace("bronze/", "", 1)

bronze_path = (
    "abfss://bronze@stcommercesrc.dfs.core.windows.net/"
    + relative_path
)

print("Bronze path:")
print(bronze_path)

In [0]:
print("STEP 1 - Reading Bronze order file")

df_bronze = spark.read.parquet(bronze_path)

rows_read = df_bronze.count()

print("Rows read from Bronze:", rows_read)

df_bronze.printSchema()

display(df_bronze.limit(10))

In [0]:
from pyspark.sql.functions import (
    col, trim, upper, when, row_number
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
print("STEP 22.3 - Cleaning Reviews")

df_clean = (
    df_bronze
    .withColumn("review_text", trim(col("review_text")))
    .withColumn("review_status", upper(trim(col("review_status"))))
    .withColumn("rating", col("rating").cast("int"))
)

print("Bronze rows:", df_bronze.count())
display(df_clean.limit(10))

In [0]:
print("STEP 22.4 - Deduplicating Reviews")

window_spec = (
    Window
    .partitionBy("review_id")
    .orderBy(
        col("updated_at").desc_nulls_last(),
        col("created_at").desc_nulls_last()
    )
)

df_deduplicated = (
    df_clean
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

print("Rows after deduplication:", df_deduplicated.count())

In [0]:
print("STEP 22.5 - Checking Review Statuses")

df_deduplicated.groupBy("review_status").count().orderBy(
    col("count").desc()
).show(truncate=False)

df_deduplicated.groupBy("rating").count().orderBy(
    "rating"
).show()

In [0]:
from pyspark.sql.functions import col

print("STEP 22.6 - Validating Reviews")

valid_review_statuses = ["APPROVED", "PENDING", "REJECTED"]

df_valid = df_deduplicated.filter(
    col("review_id").isNotNull()
    & col("customer_id").isNotNull()
    & col("product_id").isNotNull()
    & col("rating").between(1, 5)
    & col("review_status").isin(valid_review_statuses)
    & col("review_date").isNotNull()
)

df_rejected = df_deduplicated.join(
    df_valid.select("review_id"),
    on="review_id",
    how="left_anti"
)

print("Deduplicated rows:", df_deduplicated.count())
print("Valid rows:", df_valid.count())
print("Rejected rows:", df_rejected.count())

display(df_rejected)

In [0]:
print("STEP 22.7 - Saving Rejected Reviews")

exception_path = (
    "abfss://silver@stcommercesrc.dfs.core.windows.net/"
    "exceptions/reviews"
)

(
    df_rejected
    .withColumn("exception_reason",
        when(
            col("rating").isNull() | ~col("rating").between(1, 5),
            "INVALID_RATING"
        ).otherwise("INVALID_REQUIRED_FIELD")
    )
    .write
    .format("delta")
    .mode("append")
    .save(exception_path)
)

print("Rejected reviews saved to:", exception_path)

In [0]:
print("STEP 22.8 - Merging Reviews into Silver")

target_table = "ecommerce.silver.reviews_external"

target = DeltaTable.forName(spark, target_table)

(
    target.alias("t")
    .merge(
        df_valid.alias("s"),
        "t.review_id = s.review_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Reviews successfully merged into Silver.")

In [0]:
%sql
SELECT * FROM ecommerce.silver.reviews_external

review_id,customer_id,product_id,rating,review_text,review_status,review_date,created_at,updated_at
2,24133,840,5,Dolor magnam ad dicta officiis molestias dolore nihil repellat ab rem.,APPROVED,2025-10-18T12:12:24.000Z,2025-10-18T12:12:24.000Z,2025-10-31T12:12:24.000Z
3,1042,245,2,Eius dolor quis aut fugit facere exercitationem eaque aliquam itaque commodi.,APPROVED,2025-08-11T00:47:04.000Z,2025-08-11T00:47:04.000Z,2025-08-11T00:47:04.000Z
4,18391,1629,5,Mollitia repudiandae nulla similique neque provident aliquid dolorem enim aspernatur doloremque aut eius.,APPROVED,2026-09-14T12:02:01.000Z,2026-09-14T12:02:01.000Z,2026-09-21T12:02:01.000Z
8,19783,2167,5,Consequuntur quia molestiae quia incidunt nesciunt facere voluptatum consequuntur laudantium quidem similique corrupti cum perspiciatis nisi harum.,APPROVED,2026-07-14T16:19:50.000Z,2026-07-14T16:19:50.000Z,2026-07-17T16:19:50.000Z
11,9483,654,5,Ut soluta rerum esse animi soluta.,APPROVED,2024-05-02T08:16:32.000Z,2024-05-02T08:16:32.000Z,2024-05-16T08:16:32.000Z
12,20831,2989,3,Quasi repellendus voluptate minus commodi optio veniam cumque minus.,APPROVED,2026-07-31T07:27:37.000Z,2026-07-31T07:27:37.000Z,2026-08-08T07:27:37.000Z
13,22998,585,5,Vitae ut quidem porro quae id perferendis fuga tempore iste dolorum.,APPROVED,2024-04-09T11:03:30.000Z,2024-04-09T11:03:30.000Z,2024-04-16T11:03:30.000Z
35,16960,3699,3,Impedit ex iure aperiam rerum harum quo.,APPROVED,2024-01-31T08:12:14.000Z,2024-01-31T08:12:14.000Z,2024-01-31T08:12:14.000Z
36,19278,4538,4,Fugit ab explicabo quidem sequi veritatis in quibusdam molestias nobis dolore dolor ipsa adipisci.,APPROVED,2025-06-06T03:49:03.000Z,2025-06-06T03:49:03.000Z,2025-06-28T03:49:03.000Z
37,20680,483,4,Quam saepe ullam voluptatem modi tenetur nihil optio.,PENDING,2024-05-10T09:47:17.000Z,2024-05-10T09:47:17.000Z,2024-05-12T09:47:17.000Z
